# Matplotlib

This module introduces matplotlib, the foundational plotting library for
Python and the engine behind almost every other charting tool in the
ecosystem (pandas plotting, seaborn, the static side of plotly). Readers
are assumed to know Python and pandas; the focus here is on producing
useful charts directly from a DataFrame and on the small set of matplotlib
concepts (Figure, Axes, the pyplot wrapper) that explain why otherwise
identical-looking calls behave differently.

The topics below are arranged linearly for review. Structural grouping
(sections, chapters) can be applied later. The same running example, a
two year monthly Sales Plan across four regions, is reused throughout, so
each topic introduces exactly one new chart or styling idea.

---

## Topic list

1. Matplotlib in the plotting landscape
2. The running example
3. Figure and Axes
4. The pandas plotting interface
5. Line plots over time
6. Bar plots, grouped and stacked
7. Histograms and density
8. Box and violin plots
9. Scatter plots and pairwise structure
10. Correlation matrices and heatmaps
11. Cross correlation and autocorrelation
12. Faceting with subplots
13. Titles, labels, ticks, legends
14. Color and palettes
15. Saving figures for reports
16. Real world design principles
17. Common mistakes

---

## 1. Matplotlib in the plotting landscape

Matplotlib was created by John Hunter in 2003 as a Python port of
MATLAB's plotting interface. Two decades later it is still the default
charting library in scientific Python, and almost every higher level
plotting tool either wraps it (pandas `df.plot`, seaborn) or copies its
mental model (plotly's static export, bokeh's figure layout).

Two design facts shape everything that follows. First, matplotlib is a
**state machine wrapper around an object oriented core**. The pyplot
module (`import matplotlib.pyplot as plt`) tracks a current figure and a
current axes; calls like `plt.plot(...)` mutate that current axes. The
underlying objects (`Figure`, `Axes`) are also directly available, and
production code usually works with them explicitly. Second, matplotlib is
**rendering agnostic**. The same plotting code drives a screen window, a
PNG file, an SVG file, or a notebook inline display, depending on the
configured backend.

For tm1py work, matplotlib is the right tool when the output is a static
chart for a slide, a PDF, or an email. It is not the right tool for
interactive dashboards (use plotly or a BI tool) or for the kind of
high cardinality streaming visualisation that web frameworks specialise
in. The rough rule is: if the chart is going to be looked at once and
shared as an image, matplotlib produces it with the least ceremony.

## 2. The running example

The examples in this module use a synthetic monthly Sales Plan covering
the years 2025 and 2026 across four regions. The DataFrame has a
seasonal pattern (a sine wave over the calendar year), a small upward
trend, and per row noise so histograms and correlation plots have
something to show.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)

periods: pd.DatetimeIndex = pd.date_range("2025-01-01", periods=24, freq="MS")
regions: list[str] = ["Europe", "Americas", "Asia", "Pacific"]
base_revenue: dict[str, float] = {
    "Europe":   120_000.0,
    "Americas": 250_000.0,
    "Asia":      90_000.0,
    "Pacific":   60_000.0,
}

def build_region(region: str) -> pd.DataFrame:
    seasonal = 1.0 + 0.20 * np.sin(2 * np.pi * (periods.month - 1) / 12)
    trend    = 1.0 + 0.005 * np.arange(len(periods))
    noise    = rng.normal(1.0, 0.04, len(periods))
    revenue  = base_revenue[region] * seasonal * trend * noise
    units    = revenue / rng.normal(100.0, 4.0, len(periods))
    cost     = revenue * rng.normal(0.70, 0.015, len(periods))
    return pd.DataFrame({
        "period":  periods,
        "region":  region,
        "revenue": revenue,
        "units":   units,
        "cost":    cost,
    })

sales: pd.DataFrame = pd.concat([build_region(r) for r in regions], ignore_index=True)

sales.head()
#       period   region        revenue        units          cost
# 0 2025-01-01   Europe  102_837.421...  1_032.18...  72_124.19...
# 1 2025-02-01   Europe  108_551.013...  1_088.32...  76_185.71...
# ...

The shape is 96 rows by 5 columns: 24 monthly periods times 4 regions.
This is the same Sales Plan shape used in the pandas module, extended
with two years of history so seasonality, year on year trend, and
between region correlation are all present in the data.

## 3. Figure and Axes

Two objects sit at the centre of matplotlib's model. A `Figure` is the
whole canvas: one window, one PNG, one PDF page. An `Axes` is a single
plot inside that canvas, with its own data area, axis lines, ticks, and
legend. A figure may contain one Axes (the common case) or a grid of
them (faceting, Topic 12).

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([0, 1, 2, 3], [10, 11, 9, 12])
ax.set_title("Toy line")
ax.set_xlabel("step")
ax.set_ylabel("value")
fig.tight_layout()
plt.show()

`plt.subplots()` returns the pair `(Figure, Axes)`; with arguments it
returns `(Figure, ndarray of Axes)`. This explicit form is the style
used throughout the rest of the module. The pyplot only form
(`plt.plot(...)`, `plt.title(...)`, no `fig`/`ax` variables) is shorter
for one off interactive use but loses the ability to address a specific
Axes once more than one is on the canvas.

A useful mental model: every styling method on the Axes is named
`set_<thing>` (`set_title`, `set_xlabel`, `set_xscale`, `set_xlim`); the
Figure owns layout (`tight_layout`, `savefig`, `suptitle`). When a call
"feels like" it should be on the figure but is not working, try the
Axes, and vice versa. Most styling lives on the Axes.

## 4. The pandas plotting interface

A DataFrame's `.plot` method is a thin wrapper around matplotlib. It
reads the DataFrame's index as the x axis and each column as a separate
series, returns the underlying Axes, and accepts a `kind=` argument that
selects the chart type.

In [ ]:
monthly_total = sales.groupby("period")["revenue"].sum()

ax = monthly_total.plot(figsize=(9, 4), title="Total revenue per month")
ax.set_ylabel("revenue")
ax.figure.tight_layout()
plt.show()

Two facts are worth absorbing early. First, the return value is an Axes
(or an array of Axes for faceted variants), and any further styling uses
the same `set_*` methods as direct matplotlib code. The pandas wrapper
does not hide matplotlib; it just builds the first call for you.
Second, the index of the Series or DataFrame becomes the x axis without
asking, which is why a deliberate `set_index("period")` or
`groupby("period")` upstream of the plot is usually the cleanest
approach.

The `kind=` argument covers `"line"` (default), `"bar"`, `"barh"`,
`"hist"`, `"box"`, `"kde"`, `"area"`, `"scatter"`, `"hexbin"`, and
`"pie"`. Equivalent calls exist as named methods: `df.plot.bar(...)` is
identical to `df.plot(kind="bar", ...)` and reads better in a chain. The
rest of this module uses the named method form.

## 5. Line plots over time

A line plot is the default for time series and the right choice whenever
the x axis represents an ordered continuum (dates, sequence, frequency).
Multiple columns plotted against the same index produce one line per
column with an automatic legend.

In [ ]:
by_region = sales.pivot_table(
    index="period",
    columns="region",
    values="revenue",
    aggfunc="sum",
)

fig, ax = plt.subplots(figsize=(9, 4))
by_region.plot(ax=ax, linewidth=1.6)
ax.set_title("Monthly revenue by region")
ax.set_ylabel("revenue")
ax.set_xlabel("")
ax.legend(title="region", loc="upper left")
ax.grid(True, alpha=0.3)
fig.tight_layout()

Passing `ax=ax` into `plot` is the pattern that lets the pandas wrapper
draw onto a pre created Axes; without it, pandas creates its own Figure
and Axes and the explicit ones go unused. This matters as soon as
faceting (Topic 12) or annotations enter the picture.

A few line plot defaults are worth changing more often than not.
`linewidth=1.6` (or higher) renders better than the matplotlib default
of `1.0` when the chart will be projected or printed. `alpha=0.3` on
the grid keeps it visible without competing with the data. Setting
`xlabel=""` strips the redundant "period" axis label when the date
ticks already speak for themselves.

## 6. Bar plots, grouped and stacked

Bar plots compare discrete categories. The same DataFrame can be drawn
in three shapes: one bar per category (a single Series), grouped bars
(one bar per category per series), or stacked bars. Pandas selects
between them based on `stacked=` and the DataFrame shape.

In [ ]:
total_by_region = sales.groupby("region")["revenue"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
total_by_region.plot.bar(ax=ax, color="#4C72B0")
ax.set_title("Total revenue by region, two years combined")
ax.set_ylabel("revenue")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()

Grouped and stacked variants take a wide DataFrame, where each column
becomes a series in the legend.

In [ ]:
yearly = (
    sales.assign(year=sales["period"].dt.year)
         .pivot_table(index="region", columns="year", values="revenue", aggfunc="sum")
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
yearly.plot.bar(ax=axes[0], rot=0)
axes[0].set_title("Grouped: revenue by region and year")
axes[0].set_xlabel("")

yearly.plot.bar(ax=axes[1], rot=0, stacked=True)
axes[1].set_title("Stacked: revenue by region and year")
axes[1].set_xlabel("")
fig.tight_layout()

The choice between grouped and stacked is editorial. Grouped bars make
year on year comparison within a region easy; stacked bars make total
revenue per region easy and year on year comparison harder. Pick based
on the question the chart is meant to answer, not on which one looks
denser.

## 7. Histograms and density

A histogram bins a numeric column and plots the count per bin. It is
the standard first chart for understanding the distribution of a
measure: where its mass sits, whether it is skewed, whether it is
multimodal.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sales["revenue"].plot.hist(ax=ax, bins=30, color="#4C72B0", edgecolor="white")
ax.set_title("Distribution of monthly revenue across all regions")
ax.set_xlabel("revenue")
ax.set_ylabel("count")
fig.tight_layout()

The bin count is the only parameter that materially changes a
histogram's appearance and the only one worth tuning by hand. Too few
bins hide structure, too many produce a comb of noise. A reasonable
default for 100 to a few thousand observations is `bins=30`; for larger
samples, `bins="auto"` lets matplotlib pick.

For per group comparison, the cleanest form is one histogram per
region, drawn with transparency on the same Axes. The pandas wrapper
draws this directly when given `by=`:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)
for ax, (region, group) in zip(axes.flat, sales.groupby("region")):
    group["revenue"].plot.hist(ax=ax, bins=15, color="#4C72B0", edgecolor="white")
    ax.set_title(region)
    ax.set_xlabel("revenue")
fig.suptitle("Revenue distribution per region")
fig.tight_layout()

The companion plot type is the kernel density estimate (`plot.kde()`),
which draws a smoothed continuous estimate of the distribution. KDEs
read better than histograms when comparing several distributions on the
same Axes; histograms read better when the bin counts themselves matter
(as they often do when the audience is not statistically trained).

## 8. Box and violin plots

A box plot summarises a distribution with five numbers (min, first
quartile, median, third quartile, max) plus outliers. A violin plot
adds a density trace along each side of the box. Both are the right
tool when the goal is to compare distributions across categories on a
single Axes.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sales.boxplot(column="revenue", by="region", ax=ax, grid=False)
ax.set_title("Monthly revenue by region")
ax.set_ylabel("revenue")
ax.set_xlabel("")
fig.suptitle("")
fig.tight_layout()

`fig.suptitle("")` is the boilerplate fix for a quirk of pandas
`boxplot`: it sets a figure level title ("Boxplot grouped by region")
that overlaps with the Axes title set on the next line. Clearing the
suptitle removes the duplicate.

For violin plots, drop down to matplotlib directly because the pandas
wrapper does not expose them:

In [ ]:
groups = [g["revenue"].to_numpy() for _, g in sales.groupby("region")]
labels = [r for r, _ in sales.groupby("region")]

fig, ax = plt.subplots(figsize=(8, 4))
parts = ax.violinplot(groups, showmedians=True)
ax.set_xticks(range(1, len(labels) + 1))
ax.set_xticklabels(labels)
ax.set_title("Revenue distribution by region (violin)")
ax.set_ylabel("revenue")
fig.tight_layout()

`violinplot` takes a list of arrays, not a DataFrame; the small loop
that builds `groups` and `labels` is the standard adapter. Box and
violin charts are the right answer when "distribution per category" is
the question; bar plots of means hide too much.

## 9. Scatter plots and pairwise structure

A scatter plot draws one point per row, with two columns mapped to x
and y. It is the standard chart for inspecting the relationship between
two numeric measures.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sales.plot.scatter(x="units", y="revenue", ax=ax, alpha=0.6, c="#4C72B0")
ax.set_title("Revenue vs. units shipped (all regions)")
fig.tight_layout()

Two enhancements turn the scatter from a sanity check into a chart that
communicates structure. First, encode a third variable in colour using
`c=` with a column or a category Series; second, encode a fourth in
size using `s=`. For example, colouring points by region surfaces that
the slope of revenue per unit varies by region.

In [ ]:
region_to_color = {"Europe": "#4C72B0", "Americas": "#DD8452",
                   "Asia": "#55A868", "Pacific": "#C44E52"}

fig, ax = plt.subplots(figsize=(7, 5))
for region, group in sales.groupby("region"):
    ax.scatter(group["units"], group["revenue"],
               label=region, alpha=0.7, color=region_to_color[region])
ax.set_xlabel("units")
ax.set_ylabel("revenue")
ax.set_title("Revenue vs. units, coloured by region")
ax.legend(title="region")
fig.tight_layout()

The looped form (one `scatter` call per region) is more verbose than a
single `plot.scatter` with a colour column, but it produces the legend
automatically and gives full control over per group styling. For
production charts this is usually the right form.

For a quick look at every pairwise relationship at once, pandas exposes
`pd.plotting.scatter_matrix(df.select_dtypes("number"))`, which draws
an N by N grid of scatter plots with histograms on the diagonal. This
is a diagnostic tool, not a presentation chart.

## 10. Correlation matrices and heatmaps

Pearson correlation between columns is one method call: `df.corr()`.
Plotting the resulting square matrix as a heatmap gives a fast visual
read of which measures move together. Matplotlib draws the heatmap
with `imshow`; the colour annotations are added with a small loop.

In [ ]:
numeric = sales[["revenue", "units", "cost"]]
corr: pd.DataFrame = numeric.corr()
print(corr)
#          revenue     units      cost
# revenue   1.0000    0.6231    0.9994
# units     0.6231    1.0000    0.6225
# cost      0.9994    0.6225    1.0000

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1.0, vmax=1.0)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=30, ha="right")
ax.set_yticklabels(corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.values[i, j]:+.2f}",
                ha="center", va="center", color="black", fontsize=9)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Pearson correlation between measures")
fig.tight_layout()

`vmin=-1.0` and `vmax=1.0` anchor the colour scale to the meaningful
range for correlation; without them, matplotlib would stretch the
colour scale to fit the actual values, which makes near identical
correlations look as visually different as opposites. `cmap="RdBu_r"`
is a diverging palette centred on zero, which is the right choice
whenever the data has a meaningful midpoint (zero correlation, zero
deviation from a baseline, zero year on year change). For data with no
meaningful midpoint (revenue, counts), use a sequential palette
(`"viridis"`, `"plasma"`).

## 11. Cross correlation and autocorrelation

For time series, two specialised correlation plots answer questions
that the static matrix above cannot. Autocorrelation measures how
similar a series is to a lagged copy of itself; it surfaces seasonality
and persistence. Cross correlation measures how similar one series is
to a lagged copy of another; it surfaces lead and lag relationships
between two measures.

Matplotlib provides both directly on the Axes:

In [ ]:
europe = (
    sales[sales["region"] == "Europe"]
    .set_index("period")["revenue"]
    .sort_index()
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.acorr(europe.values - europe.values.mean(), maxlags=12, usevlines=True)
ax.set_title("Autocorrelation of Europe revenue")
ax.set_xlabel("lag (months)")
ax.set_ylabel("correlation")
ax.axhline(0.0, color="black", linewidth=0.6)
fig.tight_layout()

Two details matter for `acorr`. The series must be passed as a NumPy
array, and it should be mean centred (subtract the mean) before the
call; matplotlib's `acorr` does not centre the input on its own and a
constant offset distorts the lag zero spike. `maxlags=12` is the right
window for monthly data with annual seasonality: it shows the spike at
lag 12 if seasonality is present. `usevlines=True` draws the
correlation as vertical lines from zero, the conventional ACF style.

Cross correlation works the same way, with two series.

In [ ]:
americas = (
    sales[sales["region"] == "Americas"]
    .set_index("period")["revenue"]
    .sort_index()
)

x = europe.values - europe.values.mean()
y = americas.values - americas.values.mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.xcorr(x, y, maxlags=12, usevlines=True, normed=True)
ax.set_title("Cross correlation: Europe vs. Americas revenue")
ax.set_xlabel("lag (months); positive = Americas leads Europe")
ax.set_ylabel("correlation")
ax.axhline(0.0, color="black", linewidth=0.6)
fig.tight_layout()

The sign convention catches everyone at first. Matplotlib's `xcorr(x,
y, ...)` plots the correlation of `x` with `y` shifted by the lag, so a
peak at positive lag means `y` leads `x` (the future of `y` predicts
the present of `x`, or equivalently the past of `x` predicts the
present of `y`). Annotating the x axis with the interpretation, as
above, removes the ambiguity for the reader.

For a ready made autocorrelation plot, pandas offers
`pd.plotting.autocorrelation_plot(series)`, which returns an Axes and
draws the ACF along with the 95% and 99% confidence bands. It is
shorter than the matplotlib form but less tunable; for production
charts the matplotlib version usually wins.

## 12. Faceting with subplots

Faceting splits a single dataset across a grid of small Axes, one per
group, with shared axes so the comparison is visual. Matplotlib's
`subplots(nrows, ncols)` returns the grid; iterating over a `groupby`
and an Axes array in lockstep is the standard pattern.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)

for ax, (region, group) in zip(axes.flat, sales.groupby("region")):
    series = group.set_index("period")["revenue"].sort_index()
    series.plot(ax=ax, color="#4C72B0", linewidth=1.5)
    ax.set_title(region)
    ax.set_xlabel("")
    ax.set_ylabel("revenue")
    ax.grid(True, alpha=0.3)

fig.suptitle("Monthly revenue per region")
fig.tight_layout()

`sharex=True` and `sharey=True` are what make the grid useful: every
panel reads on the same scale, so the comparison between regions is
honest. Without sharing, each panel auto scales to its own data and
small regions look as tall as large ones, which is misleading.

`axes.flat` flattens the 2 by 2 array of Axes into an iterable of
length 4; `zip` then pairs each Axes with one (region, group) tuple
from the `groupby`. When the number of groups does not match the grid
size, the trailing empty Axes need explicit cleanup
(`fig.delaxes(axes.flat[i])`) or they render as blank boxes. The
cleanest mitigation is to pick the grid shape to match the group count
exactly.

## 13. Titles, labels, ticks, legends

The default matplotlib chart is functional but rarely presentation
ready. The set of styling calls that turn a draft chart into a
shareable one is small and worth committing to memory.

In [ ]:
ts = sales.groupby("period")["revenue"].sum()

fig, ax = plt.subplots(figsize=(9, 4))
ts.plot(ax=ax, color="#4C72B0", linewidth=1.8, label="total revenue")

ax.set_title("Total monthly revenue, 2025 to 2026", fontsize=13, loc="left")
ax.set_ylabel("revenue (USD)")
ax.set_xlabel("")

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1000:,.0f}K"))
ax.tick_params(axis="x", labelrotation=0)

ax.legend(loc="upper left", frameon=False)
ax.grid(True, axis="y", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

Five small choices in the block above each have a reason. `loc="left"`
on the title puts it where readers' eyes land first. The
`FuncFormatter` rewrites tick labels in thousands so the y axis reads
as `120K` rather than `120000`. `frameon=False` on the legend removes
a box that almost never adds information. The y axis only grid keeps
the chart structured without competing with the line. Hiding the top
and right spines is a common minimalist convention; the chart loses
nothing and gains visual breathing room.

These choices are not ideology. The point is that defaults are a
starting position, not an end state, and a small consistent set of
overrides applied on every chart in a report makes the whole report
look intentional.

## 14. Color and palettes

Colour is information, not decoration. The categorical default in
matplotlib (`tab:blue`, `tab:orange`, `tab:green`, ...) is fine for up
to about ten distinct categories. Beyond that, or when the data has
ordinal or diverging structure, an explicit palette communicates more.

In [ ]:
sequential = plt.get_cmap("viridis")          # ordered low to high
diverging  = plt.get_cmap("RdBu_r")           # negative through zero to positive
qualitative = plt.get_cmap("tab10")           # distinct categorical hues

n_regions = len(regions)
colours = [sequential(i / max(n_regions - 1, 1)) for i in range(n_regions)]
region_to_color = dict(zip(regions, colours))

The three palette families correspond to three data structures.
**Sequential** palettes (`viridis`, `plasma`, `magma`, `cividis`) map
ordered numeric data; the colour goes from light to dark or the other
way. **Diverging** palettes (`RdBu_r`, `coolwarm`, `PiYG`) map data
with a meaningful midpoint; one hue for one side, another hue for the
other, neutral at the middle. **Qualitative** palettes (`tab10`,
`Set2`, `Pastel1`) map categories with no inherent order; the hues are
distinct but unranked.

A common mistake is using a qualitative palette for ordinal data
(months, quarters, years), which loses the ordering information in the
chart even though the data has it. The reverse mistake (sequential
palette for unordered categories) is rarer but reads as if there is an
ordering when there is not. Match the palette to the data, not to the
default.

For accessibility, `viridis` and `cividis` were specifically designed
to be perceptually uniform and to remain distinguishable for the most
common forms of colour blindness. They are safer defaults than the
older `jet` palette (which is still available but no longer
recommended).

## 15. Saving figures for reports

A chart that cannot be embedded in a slide or a PDF is half done.
`Figure.savefig` writes the figure to disk in a format inferred from
the file extension. The defaults produce a low resolution PNG suitable
for the screen but not for print.

In [ ]:
from pathlib import Path

fig, ax = plt.subplots(figsize=(8, 4))
sales.groupby("period")["revenue"].sum().plot(ax=ax, color="#4C72B0", linewidth=1.6)
ax.set_title("Total monthly revenue")
fig.tight_layout()

out_dir = Path("out")
out_dir.mkdir(exist_ok=True)
fig.savefig(out_dir / "revenue.png", dpi=200, bbox_inches="tight")
fig.savefig(out_dir / "revenue.pdf", bbox_inches="tight")
fig.savefig(out_dir / "revenue.svg", bbox_inches="tight")
plt.close(fig)

`dpi=200` produces a raster image sharp enough for slides; `dpi=300` is
the print convention. `bbox_inches="tight"` trims the surrounding
whitespace, which otherwise leaves a thick border around the chart in
the saved file. PDF and SVG are vector formats and ignore `dpi`; they
scale to any size without loss and are the right choice when the chart
will be resized in a slide deck or rendered at print resolution.

`plt.close(fig)` is the discipline that prevents matplotlib's
"too many open figures" warning in scripts that produce many charts in
a loop. Each `subplots` call allocates a Figure, and Python keeps it
alive until either the script exits or the figure is closed
explicitly. In a notebook, where each cell's figures usually display
inline and do not pile up, the close call is optional.

## 16. Real world design principles

Once the mechanics are in hand, a small set of guidelines applies to
almost every chart drawn for actual readers.

**Pick the chart from the question, not the data.** A bar chart, a
line, and a heatmap can all be built from the same DataFrame; only one
of them answers any given question. Ask "what should the reader
conclude from this chart?" before choosing a chart type. If the answer
is "trend over time", line. If it is "comparison between categories",
bar. If it is "shape of a distribution", histogram or box. If it is
"relationship between two measures", scatter. If it is "structure in a
matrix", heatmap.

**One chart, one message.** Charts that try to communicate two things
usually communicate neither. A grouped bar plot of four metrics across
ten categories is a small table in disguise and a real table reads
better. When the second message is genuinely needed, draw a second
chart.

**Index thoughtfully before plotting.** The pandas plotting interface
takes the index for the x axis and the columns for the series. Setting
the index deliberately upstream of the plot (with `set_index` or
`groupby`) is more efficient than fighting matplotlib's defaults
afterward.

**Match the palette to the data structure.** Sequential for ordered,
diverging for centred, qualitative for unordered. The most common
mistake is using the qualitative default for ordinal data; the chart
displays the values correctly but loses the ordering as a visual cue.

**Annotate ambiguous axes.** Cross correlation lag direction, log
scaled axes, normalised counts, and indexed time series all carry
conventions that the reader may not share. A short axis label or a
note in the title removes the guesswork.

**Save in the right format for the destination.** Raster (PNG) for
screens at known sizes, vector (PDF, SVG) for documents that may be
rescaled or printed. `dpi=200` for slides, `dpi=300` for print, and
always `bbox_inches="tight"`.

**Build a small house style and apply it everywhere.** A consistent
font size, palette, grid style, and figure size across an entire report
reads as deliberate; a per chart mix of styles reads as casual. The
five line styling block in Topic 13 is the kind of thing to lift into a
helper function and call on every Axes in the project.

## 17. Common mistakes

A short collection of errors that come up often when plotting pandas
DataFrames, each with the cleaner alternative.

**Calling `plt.show()` between independent figures in a script and
expecting them to render together.** `plt.show()` is interactive: it
blocks until the window is closed and resets pyplot state. In scripts
that produce multiple charts, save each figure with `savefig` and
`close` it; in notebooks, let each cell display its figure on its own.

In [ ]:
# Wrong (in a notebook): both figures share state, only the second is styled
plt.plot(ts1)
plt.title("first")
plt.plot(ts2)
plt.title("second")

# Correct
fig, ax = plt.subplots()
ax.plot(ts1)
ax.set_title("first")

fig, ax = plt.subplots()
ax.plot(ts2)
ax.set_title("second")

**Using `ax = df.plot(...)` and then drawing on `plt`.** The pyplot
calls (`plt.title`, `plt.xlabel`) target the current Axes, which may
or may not be the one returned. Once an `ax` variable exists, use it.

In [ ]:
# Wrong: title may land on the wrong Axes after a later plot call
ax = df.plot()
plt.title("revenue")

# Correct
ax = df.plot()
ax.set_title("revenue")

**Forgetting to centre a series before `acorr` / `xcorr`.** A constant
offset distorts the lag zero spike and shifts every other lag.

In [ ]:
# Wrong: raw revenue, mean is large and positive
ax.acorr(europe.values, maxlags=12)

# Correct
ax.acorr(europe.values - europe.values.mean(), maxlags=12)

**Using a qualitative palette for ordered data.** The default colours
for categorical groupings (`tab:blue`, `tab:orange`, ...) carry no
ordering. Plotting months, quarters, or years with them strips the
order from the chart even though the data has it.

In [ ]:
# Wrong: months coloured with the categorical default
ax = sales.pivot_table(index="region", columns="month", values="revenue").plot.bar()

# Correct: sequential palette respects the calendar
month_colours = plt.get_cmap("viridis")(np.linspace(0, 1, 12))
ax = sales.pivot_table(index="region", columns="month", values="revenue").plot.bar(color=list(month_colours))

**Letting `boxplot` set a duplicate figure title.** Pandas `boxplot`
calls `fig.suptitle("Boxplot grouped by ...")` automatically, which
overlaps with any Axes title set after.

In [ ]:
# Wrong: two stacked titles
sales.boxplot(column="revenue", by="region", ax=ax)
ax.set_title("Revenue by region")

# Correct: clear the suptitle that boxplot added
sales.boxplot(column="revenue", by="region", ax=ax)
ax.set_title("Revenue by region")
ax.figure.suptitle("")

**Stretching the colour scale of a correlation heatmap to fit the
data.** Without `vmin=-1.0, vmax=1.0`, a heatmap of correlations near
0.9 looks visually as different from one near 0.95 as a heatmap of 1.0
versus −1.0. The colour scale must match the meaning.

In [ ]:
# Wrong: scale fits the data, not the meaning
ax.imshow(corr.values, cmap="RdBu_r")

# Correct
ax.imshow(corr.values, cmap="RdBu_r", vmin=-1.0, vmax=1.0)

**Saving figures without `bbox_inches="tight"`.** The default leaves a
generous margin of whitespace around the chart, which then sits inside
whatever the slide or document adds on top.

In [ ]:
# Wrong: thick whitespace border in the saved file
fig.savefig("out/chart.png", dpi=200)

# Correct
fig.savefig("out/chart.png", dpi=200, bbox_inches="tight")